# Transformaciones avanzadas con Polars

Notebook autónomo para practicar un flujo analítico reproducible sin pandas ni NumPy. Se usan datos sintéticos de ventas, sensores y tarifas para cubrir **LazyFrame**, optimización, ventanas, agregaciones condicionales, `join_asof` y ventanas temporales dinámicas.

> Las salidas se calculan al ejecutar cada bloque; el propósito y la interpretación acompañan a cada resultado.

In [ ]:
from datetime import datetime, timedelta
import polars as pl

pl.Config.set_tbl_rows(12)

# Datos deterministas: tres tiendas, ocho horas y una lectura por hora.
base = datetime(2026, 1, 5, 8, 0)
ventas = pl.DataFrame({
    "tienda": [t for t in ("Centro", "Norte", "Sur") for _ in range(8)],
    "instante": [base + timedelta(hours=h) for h in range(8)] * 3,
    "producto": (["A", "B", "A", "C", "B", "A", "C", "B"] * 3),
    "unidades": ([4, 2, 7, 1, 5, 3, 6, 2] * 3),
    "precio": ([12.0, 25.0, 12.0, 40.0, 25.0, 12.0, 40.0, 25.0] * 3),
})
ventas = ventas.with_columns((pl.col("unidades") * pl.col("precio")).alias("importe"))
tarifas = pl.DataFrame({
    "tienda": ["Centro", "Norte", "Sur"],
    "desde": [base - timedelta(hours=1)] * 3,
    "zona": ["urbana", "residencial", "industrial"],
    "factor": [1.00, 0.95, 1.08],
})
ventas.head(6)

## 1. LazyFrame y plan optimizado

**Propósito:** construir una consulta diferida que seleccione pronto las columnas necesarias y filtre antes de agregar. `explain(optimized=True)` permite inspeccionar el plan que Polars ejecutará, sin materializar todavía el resultado.

In [ ]:
consulta = (
    ventas.lazy()
    .filter(pl.col("unidades") >= 3)
    .select(["tienda", "producto", "unidades", "importe"])
    .group_by(["tienda", "producto"])
    .agg(
        pl.len().alias("operaciones"),
        pl.col("unidades").sum().alias("unidades_validas"),
        pl.col("importe").sum().round(2).alias("ingresos")
    )
    .sort(["tienda", "ingresos"], descending=[False, True])
)
print(consulta.explain(optimized=True))
resultado_lazy = consulta.collect()
resultado_lazy

**Interpretación:** el plan se resuelve al llamar a `collect()`. El filtro reduce las filas antes del `group_by`; esto evita cargar o agrupar registros que no participan en el análisis.

## 2. Expresiones reutilizables y agregación condicional

**Propósito:** encapsular reglas de negocio en expresiones Polars composables. La clasificación de importe y el cálculo de unidades/ingresos de productos concretos se realizan dentro de una sola agregación.

In [ ]:
importe_alto = pl.col("importe") >= 100
categoria = (
    pl.when(importe_alto).then(pl.lit("alto"))
    .when(pl.col("importe") >= 50).then(pl.lit("medio"))
    .otherwise(pl.lit("bajo"))
    .alias("segmento_importe")
)

clasificado = ventas.with_columns(categoria)
resumen_condicional = (
    clasificado.group_by("tienda")
    .agg(
        pl.col("importe").sum().round(2).alias("ingresos_totales"),
        pl.col("importe").filter(pl.col("segmento_importe") == "alto").sum().round(2).alias("ingresos_altos"),
        pl.col("unidades").filter(pl.col("producto") == "A").sum().alias("unidades_producto_A"),
        pl.col("producto").n_unique().alias("productos_distintos"),
    )
    .sort("ingresos_totales", descending=True)
)
resumen_condicional

## 3. Ventanas (`over`) para comparar cada fila con su tienda

**Propósito:** añadir métricas de contexto sin perder el grano original. La cuota de cada venta respecto al total de su tienda y el ranking de importe se calculan con expresiones de ventana.

In [ ]:
con_contexto = (
    ventas.with_columns([
        pl.col("importe").sum().over("tienda").alias("ingresos_tienda"),
        pl.col("importe").rank("dense", descending=True).over("tienda").alias("ranking_en_tienda"),
    ])
    .with_columns(
        (pl.col("importe") / pl.col("ingresos_tienda") * 100).round(1).alias("cuota_tienda_pct")
    )
    .sort(["tienda", "ranking_en_tienda", "instante"])
)
con_contexto.select(["tienda", "instante", "importe", "ranking_en_tienda", "cuota_tienda_pct"]).head(9)

**Interpretación:** `over("tienda")` mantiene una fila por venta, pero calcula contra el particionado de cada tienda. Es distinto de un `group_by`, que cambiaría el número de filas.

## 4. `join_asof`: enriquecer hechos con la tarifa vigente

**Propósito:** combinar cada venta con la última tarifa conocida para la misma tienda. `join_asof` es apropiado para datos ordenados temporalmente y evita una combinación cartesiana.

In [ ]:
# El orden global por clave temporal conserva el orden dentro de cada tienda.
ventas_ordenadas = ventas.sort("instante")
tarifas_ordenadas = tarifas.sort("desde")
enriquecidas = ventas_ordenadas.join_asof(
    tarifas_ordenadas,
    left_on="instante",
    right_on="desde",
    by="tienda",
    strategy="backward",
    # Polars 2.1 no puede comprobar automáticamente sortedness con `by`;
    # por eso las entradas se ordenan explícitamente por la clave temporal.
    check_sortedness=False,
).with_columns(
    (pl.col("importe") * pl.col("factor")).round(2).alias("importe_ajustado")
)
enriquecidas.select(["tienda", "instante", "zona", "factor", "importe", "importe_ajustado"]).head(6)

**Interpretación:** `strategy="backward"` busca la tarifa cuyo instante es anterior o igual a la venta. El orden por las claves temporales es un requisito importante para que la unión sea correcta.

## 5. Agregación temporal dinámica y media móvil

**Propósito:** resumir un flujo temporal en ventanas de dos horas y calcular una media móvil por tienda. `group_by_dynamic` trabaja con ventanas de tiempo y `rolling_mean` aporta una señal suavizada.

In [ ]:
por_hora = (
    ventas.sort("instante")
    .group_by_dynamic("instante", every="2h", period="2h", group_by="tienda", closed="left")
    .agg(
        pl.col("importe").sum().round(2).alias("ingresos_2h"),
        pl.col("unidades").sum().alias("unidades_2h"),
        pl.len().alias("ventas_2h"),
    )
    .sort(["tienda", "instante"])
)
por_hora = por_hora.with_columns(
    pl.col("ingresos_2h").rolling_mean(window_size=3, min_samples=1).over("tienda").round(2).alias("media_movil_3_ventanas")
)
por_hora.head(9)

**Interpretación:** cada fila representa una ventana de dos horas; la media móvil suaviza tres ventanas consecutivas dentro de cada tienda. Este patrón resulta útil para detectar cambios de ritmo sin abandonar la dimensión temporal.

## Conclusiones

- Las consultas lazy permiten inspeccionar y optimizar el plan antes de materializar datos.
- Las expresiones y ventanas concentran reglas complejas sin bucles Python ni alterar el grano de los hechos.
- `join_asof` resuelve enriquecimientos temporales cuando las tablas están ordenadas.
- Las ventanas dinámicas y las medias móviles convierten eventos en señales temporales comparables.

Todo el ejemplo usa únicamente Polars y la biblioteca estándar, con datos sintéticos reproducibles.